# OE1 · Actividad 3 — Extracción y validación de referencias VoxForge Spanish

Extrae `etc/PROMPTS` y `etc/prompts-original` de cada sesión, asocia textos con el inventario de EDA y el dataset normalizado, genera referencias **provisionales** y un archivo para correcciones humanas. No ejecuta Whisper ni modifica los WAV. Ejecutar celdas en orden. Reejecución segura: las correcciones manuales se guardan en archivo separado y no se sobrescriben.

In [2]:
from pathlib import Path
import re, unicodedata
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == "notebooks": ROOT = ROOT.parent
RAW = ROOT / "data" / "raw" / "voxforge-es"
META = ROOT / "data" / "metadata"
REF = ROOT / "data" / "references"
REF.mkdir(parents=True, exist_ok=True)
INVENTARIO = META / "inventario_voxforge.csv"
PROCESADOS = META / "dataset_procesado.csv"
SALIDA = REF / "transcripciones_referencia.csv"
REVISIONES = REF / "correcciones_manuales.csv"
COBERTURA = META / "cobertura_referencias_voxforge.csv"
INCIDENCIAS = META / "incidencias_referencias_voxforge.csv"
for ruta in [RAW, INVENTARIO]:
    if not ruta.exists(): raise FileNotFoundError(f"Falta: {ruta}")
print("Raíz:", ROOT)
print("Referencias:", REF)


Raíz: D:\User\Escritorio\tareas\Ciclo 8\ProyectoFinal
Referencias: D:\User\Escritorio\tareas\Ciclo 8\ProyectoFinal\data\references


d:\User\Escritorio\tareas\Ciclo 8\ProyectoFinal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Inventario y correspondencia de identificadores

El EDA utiliza `ID_Audio` con `wav` en la ruta, mientras que la normalización anterior genera un identificador basado en sesión + nombre. Este notebook preserva **ambos** y usa el ID de normalización para los resultados ASR.

In [2]:
inv = pd.read_csv(INVENTARIO, dtype=str).fillna("")
requeridas = {"ID_Audio", "Sesion", "Archivo_original"}
if not requeridas.issubset(inv.columns):
    raise ValueError(f"Faltan columnas: {requeridas-set(inv.columns)}")

def id_normalizado(sesion, archivo):
    return re.sub(r"[^a-zA-Z0-9_.-]", "_", f"{sesion}__{Path(archivo).stem}")

inv["ID_Audio_EDA"] = inv["ID_Audio"]
inv["ID_Audio"] = [id_normalizado(s,a) for s,a in zip(inv.Sesion, inv.Archivo_original)]
inv["Clave"] = [f"{s}/{Path(a).stem}" for s,a in zip(inv.Sesion,inv.Archivo_original)]
if inv.ID_Audio.duplicated().any() or inv.Clave.duplicated().any():
    raise ValueError("IDs duplicados: revisar inventario antes de continuar")
print("Audios en inventario:", len(inv))
display(inv[["ID_Audio", "ID_Audio_EDA", "Sesion", "Archivo_original"]].head())


Audios en inventario: 23537


,ID_Audio,ID_Audio_EDA,Sesion,Archivo_original
0,abarzuaf-20101026-caf__es-0027,abarzuaf-20101026-caf__wav__es-0027.wav,abarzuaf-20101026-caf,es-0027.wav
1,abarzuaf-20101026-caf__es-0028,abarzuaf-20101026-caf__wav__es-0028.wav,abarzuaf-20101026-caf,es-0028.wav
2,abarzuaf-20101026-caf__es-0029,abarzuaf-20101026-caf__wav__es-0029.wav,abarzuaf-20101026-caf,es-0029.wav
3,abarzuaf-20101026-caf__es-0030,abarzuaf-20101026-caf__wav__es-0030.wav,abarzuaf-20101026-caf,es-0030.wav
4,abarzuaf-20101026-caf__es-0031,abarzuaf-20101026-caf__wav__es-0031.wav,abarzuaf-20101026-caf,es-0031.wav


## 2. Extraer PROMPTS y prompts-original

Se detecta el inicio de cada registro por un identificador de tipo `es-0027`, admitiendo rutas como `sesion/mfc/es-0027`. Las líneas sin identificador inicial se agregan al texto anterior. Los errores de lectura se registran.

In [3]:
# Identificador inicial de línea: admite rutas y el nombre de audio, con o sin .wav
PATRON = re.compile(r"^\s*(?P<clave>(?:[^\s]+/)*[A-Za-z]{2,4}-\d+(?:\.wav)?)\s+(?P<texto>.*)$")

def leer_prompts(ruta):
    if not ruta.is_file(): return {}, "archivo_ausente"
    texto = None
    for encoding in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            texto = ruta.read_text(encoding=encoding)
            break
        except UnicodeDecodeError:
            continue
    if texto is None: return {}, "codificacion_no_legible"
    registros = {}
    clave_actual = None
    duplicados = []
    for linea in texto.splitlines():
        linea = linea.strip()
        if not linea: continue
        match = PATRON.match(linea)
        if match:
            clave = Path(match.group("clave")).stem.lower()
            if clave in registros:
                duplicados.append(clave)
            else:
                registros[clave] = match.group("texto").strip()
            clave_actual = clave
        elif clave_actual is not None:
            # Continuación de una oración partida en varias líneas
            registros[clave_actual] += " " + linea
    return registros, ("ids_duplicados:" + ",".join(sorted(set(duplicados)))) if duplicados else "ok"

sesiones = sorted(inv.Sesion.unique())
textos = []
problemas_archivos = []
for sesion in tqdm(sesiones, desc="Leyendo textos de sesiones"):
    carpeta = RAW / sesion / "etc"
    original, estado_original = leer_prompts(carpeta / "prompts-original")
    prompts, estado_prompts = leer_prompts(carpeta / "PROMPTS")
    if estado_original != "ok" or estado_prompts != "ok":
        problemas_archivos.append({"Sesion":sesion, "Estado_original":estado_original, "Estado_PROMPTS":estado_prompts})
    claves = set(original) | set(prompts)
    for clave in claves:
        textos.append({"Clave": f"{sesion}/{clave}", "Texto_Original":original.get(clave,""), "Texto_PROMPTS":prompts.get(clave,"")})

df_textos = pd.DataFrame(textos, columns=["Clave","Texto_Original","Texto_PROMPTS"])
if df_textos.Clave.duplicated().any(): raise ValueError("Claves de referencia duplicadas")
print("Registros textuales encontrados:",len(df_textos))
print("Sesiones con incidencias en archivos:",len(problemas_archivos))
display(df_textos.head())


Leyendo textos de sesiones: 100%|██████████| 2277/2277 [01:17<00:00, 29.57it/s]

Registros textuales encontrados: 22579
Sesiones con incidencias en archivos: 0


,Clave,Texto_Original,Texto_PROMPTS
0,AGOs-20100831-jdm/es-0019,Camareros rubios de corta chaqueta azul y boto...,CAMAREROS RUBIOS DE CORTA CHAQUETA AZUL Y BOTO...
1,AGOs-20100831-jdm/es-0018,"Manteca y mermeladas el pan tostado, husmear e...",MANTECA Y MERMELADAS EL PAN TOSTADO HUSMEAR EL...
2,AGOs-20100831-jdm/es-0021,"Una vidriera, de pared a pared, formada de peq...",UNA VIDRIERA DE PARED A PARED FORMADA DE PEQUE...
3,AGOs-20100831-jdm/es-0022,"Dejaba ver el salón inmediato, blanco también,...",DEJABA VER EL SALóN INMEDIATO BLANCO TAMBIéN P...
4,AGOs-20100831-jdm/es-0020,"Sorteando los promontorios de los respaldos, l...",SORTEANDO LOS PROMONTORIOS DE LOS RESPALDOS LO...


## 3. Normalización de textos y control de discrepancias

Se conserva el texto original. La normalización elimina puntuación, conserva letras acentuadas y `ñ`, y comprime espacios. No modifica números escritos en palabras ni intenta corregir errores de lectura.

In [4]:
def normalizar_texto(texto):
    texto = unicodedata.normalize("NFC", str(texto)).lower()
    texto = "".join(c if (c.isalnum() or c.isspace()) else " " for c in texto)
    return " ".join(texto.split())

refs = inv.merge(df_textos, on="Clave", how="left", validate="one_to_one")
refs[["Texto_Original", "Texto_PROMPTS"]] = refs[["Texto_Original", "Texto_PROMPTS"]].fillna("")
refs["Normalizado_Original"] = refs.Texto_Original.map(normalizar_texto)
refs["Normalizado_PROMPTS"] = refs.Texto_PROMPTS.map(normalizar_texto)
refs["Tiene_Original"] = refs.Normalizado_Original.ne("")
refs["Tiene_PROMPTS"] = refs.Normalizado_PROMPTS.ne("")
refs["Discrepancia_Fuentes"] = (refs.Tiene_Original & refs.Tiene_PROMPTS & (refs.Normalizado_Original != refs.Normalizado_PROMPTS))
refs["Texto_Referencia"] = refs.Normalizado_Original.where(refs.Tiene_Original, refs.Normalizado_PROMPTS)
refs["Estado_Validacion"] = "pendiente_revision_auditiva"
refs.loc[~refs.Tiene_Original & ~refs.Tiene_PROMPTS, "Estado_Validacion"] = "sin_texto"
refs.loc[refs.Discrepancia_Fuentes, "Estado_Validacion"] = "discrepancia_fuentes"
refs.loc[~refs.Tiene_Original & refs.Tiene_PROMPTS, "Estado_Validacion"] = "solo_prompts"
refs["Observaciones"] = ""
print(refs.Estado_Validacion.value_counts(dropna=False).to_string())


Estado_Validacion
pendiente_revision_auditiva    22004
sin_texto                        983
discrepancia_fuentes             550


## 4. Incorporar correcciones manuales sin perderlas

El archivo `correcciones_manuales.csv` se crea solo la primera vez. Para audios escuchados, escribe en `Texto_Corregido_Manual` lo realmente pronunciado y marca `Validacion_Manual` como `validado` (o `excluir` cuando sea necesario). **No marques todos como validados sin escucharlos.**

In [5]:
if not REVISIONES.exists():
    plantilla = refs[["ID_Audio", "Sesion", "Archivo_original", "Texto_Original", "Texto_Referencia"]].copy()
    plantilla["Texto_Corregido_Manual"] = ""
    plantilla["Validacion_Manual"] = ""
    plantilla["Observaciones_Manual"] = ""
    plantilla.to_csv(REVISIONES, index=False, encoding="utf-8-sig")
    print("Plantilla creada:", REVISIONES)
else:
    print("Se conservó la plantilla de correcciones existente:", REVISIONES)

manual = pd.read_csv(REVISIONES, dtype=str).fillna("")
columnas_manual = {"ID_Audio", "Texto_Corregido_Manual", "Validacion_Manual", "Observaciones_Manual"}
if not columnas_manual.issubset(manual.columns):
    raise ValueError(f"Faltan columnas en correcciones: {columnas_manual-set(manual.columns)}")
if manual.ID_Audio.duplicated().any(): raise ValueError("IDs duplicados en correcciones")
refs = refs.merge(manual[list(columnas_manual)], on="ID_Audio", how="left", validate="one_to_one")
for c in columnas_manual - {"ID_Audio"}: refs[c] = refs[c].fillna("")
validos_manual = refs.Validacion_Manual.str.strip().str.lower().eq("validado")
corregidos = validos_manual & refs.Texto_Corregido_Manual.str.strip().ne("")
refs.loc[corregidos,"Texto_Referencia"] = refs.loc[corregidos,"Texto_Corregido_Manual"].map(normalizar_texto)
refs.loc[validos_manual,"Estado_Validacion"] = "validado_manualmente"
refs.loc[refs.Validacion_Manual.str.strip().str.lower().eq("excluir"),"Estado_Validacion"] = "excluir"
refs["Observaciones"] = refs.Observaciones_Manual.where(refs.Observaciones_Manual.ne(""), refs.Observaciones)


Plantilla creada: D:\User\Escritorio\tareas\Ciclo 8\ProyectoFinal\data\references\correcciones_manuales.csv


## 5. Comprobar audios normalizados y exportar

Las referencias se extraen para el corpus completo, incluso si la normalización por lotes aún no terminó. La columna `Audio_Procesado` indica si el WAV normalizado ya está disponible. El CSV final incluye estados para que no se confundan referencias extraídas con referencias auditadas.

In [1]:
if PROCESADOS.exists():
    procesados = pd.read_csv(PROCESADOS, dtype=str).fillna("")
    if "ID_Audio" not in procesados.columns:
        raise ValueError("dataset_procesado.csv no contiene ID_Audio")
    ids_procesados = set(procesados.ID_Audio)
else:
    ids_procesados = set()
refs["Audio_Procesado"] = refs.ID_Audio.isin(ids_procesados)

columnas_salida = ["ID_Audio", "ID_Audio_EDA", "Sesion", "Archivo_original", "Clave", "Texto_Original", "Texto_PROMPTS", "Normalizado_Original", "Normalizado_PROMPTS", "Texto_Referencia", "Tiene_Original", "Tiene_PROMPTS", "Discrepancia_Fuentes", "Estado_Validacion", "Audio_Procesado", "Observaciones"]
refs[columnas_salida].to_csv(SALIDA, index=False, encoding="utf-8-sig")
incidencias = refs.loc[refs.Estado_Validacion.isin(["sin_texto","discrepancia_fuentes","solo_prompts"]), columnas_salida]
incidencias.to_csv(INCIDENCIAS, index=False, encoding="utf-8-sig")

cobertura = pd.DataFrame([
    {"Indicador":"Audios inventariados", "Cantidad":len(refs)},
    {"Indicador":"Con texto de referencia", "Cantidad":int(refs.Texto_Referencia.ne("").sum())},
    {"Indicador":"Sin texto", "Cantidad":int(refs.Texto_Referencia.eq("").sum())},
    {"Indicador":"Discrepancia entre fuentes", "Cantidad":int(refs.Discrepancia_Fuentes.sum())},
    {"Indicador":"Validados manualmente", "Cantidad":int(refs.Estado_Validacion.eq("validado_manualmente").sum())},
    {"Indicador":"Audios normalizados disponibles", "Cantidad":int(refs.Audio_Procesado.sum())},
    {"Indicador":"Sesiones con incidencia de archivo", "Cantidad":len(problemas_archivos)},
])
cobertura.to_csv(COBERTURA, index=False, encoding="utf-8-sig")
print("Referencias:", SALIDA)
print("Incidencias:", INCIDENCIAS)
print("Cobertura:", COBERTURA)
display(cobertura)
display(refs[["ID_Audio", "Texto_Referencia", "Estado_Validacion"]].head(8))


NameError: name 'PROCESADOS' is not defined

## 6. Revisión auditiva para la tesis

1. Revisar primero `data/metadata/incidencias_referencias_voxforge.csv`.
2. Escuchar y validar una muestra distribuida entre sesiones; registrar criterio de muestreo y cantidad.
3. Para cada corrección, editar `data/references/correcciones_manuales.csv` sin alterar los textos fuente; volver a ejecutar celdas 4 y 5.
4. **No asumir** que el texto de lectura equivale a la pronunciación real: omisiones o repeticiones pueden inflar el WER ASR.
5. Si necesitas una medición estricta del WER del reconocedor, usa referencias fieles a la pronunciación en los audios incluidos en la evaluación.
6. La columna `Estado_Validacion` permite filtrar audios revisados o excluir incidencias según el protocolo de la tesis.